# Aprendizaje estadístico

## Carga de librerías y datos

In [ ]:
import pandas as pd
import numpy as np

# Modelado
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
batallas = pd.read_csv(
    "../data/csvs/batallas_detallado.csv"
)

batallas.head()

## Ingienería de variables

In [ ]:
df = batallas.copy()

# Diferencia de Pokémon iniciales
df["delta_num_pokemon"] = (
    df["Num_Pokemon_J1"] -
    df["Num_Pokemon_J2"]
)

# Diferencia de Pokémon vivos
df["delta_vivos"] = (
    df["Vivos_J1"] -
    df["Vivos_J2"]
)

# Diferencia de vida total
df["delta_suma_vida"] = (
    df["Suma_Vida_J1"] -
    df["Suma_Vida_J2"]
)

In [ ]:
df["delta_velocidad"] = (
    df["velocidad_activo_j1"] -
    df["velocidad_activo_j2"]
)

df["delta_ataque"] = (
    df["ataque_activo_j1"] -
    df["ataque_activo_j2"]
)

df["delta_ataque_esp"] = (
    df["ataque_esp_activo_j1"] -
    df["ataque_esp_activo_j2"]
)

df["delta_defensa"] = (
    df["defensa_activo_j1"] -
    df["defensa_activo_j2"]
)

df["delta_defensa_esp"] = (
    df["defensa_esp_activo_j1"] -
    df["defensa_esp_activo_j2"]
)

In [ ]:
variables_numericas = [

    "Turno",

    "delta_num_pokemon",
    "delta_vivos",
    "delta_suma_vida",

    "delta_velocidad",

    "delta_ataque",
    "delta_ataque_esp",

    "delta_defensa",
    "delta_defensa_esp"
]

In [ ]:
variables_categoricas = [

    "Primera_Accion",

    "Accion_J1",
    "Accion_J2"]

In [ ]:
X_cat = pd.get_dummies(

    df[variables_categoricas],

    drop_first=True
)

In [ ]:
X_num = df[variables_numericas]

In [ ]:
X = pd.concat(
    [X_num, X_cat],
    axis=1
)

Variable objetivo

In [ ]:
y = (
    df["Ganador"] == "Jugador1"
).astype(int)

Grupos para train/test

In [ ]:
groups = df["Id_Batalla"]

In [ ]:
print(X.columns.tolist())

## Modelos de clasificación para diferencias de stats

In [ ]:
batallas_ids = df["Id_Batalla"].unique()

print(len(batallas_ids))

In [ ]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(

    batallas_ids,

    test_size=0.3,

    random_state=42
)

In [ ]:
train_mask = df["Id_Batalla"].isin(train_ids)

test_mask = df["Id_Batalla"].isin(test_ids)

X_train = X[train_mask]

X_test = X[test_mask]

y_train = y[train_mask]

y_test = y[test_mask]

print(f"Batallas train: {len(train_ids)}")
print(f"Batallas test: {len(test_ids)}")

print(f"Filas train: {X_train.shape[0]}")
print(f"Filas test: {X_test.shape[0]}")

In [ ]:
from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

In [ ]:
scaler = StandardScaler()

# Escalar solo numéricas
X_train_scaled_num = scaler.fit_transform(
    X_train[variables_numericas]
)

X_test_scaled_num = scaler.transform(
    X_test[variables_numericas]
)

# Convertir a DataFrame
X_train_scaled_num = pd.DataFrame(

    X_train_scaled_num,

    columns=variables_numericas,

    index=X_train.index
)

X_test_scaled_num = pd.DataFrame(

    X_test_scaled_num,

    columns=variables_numericas,

    index=X_test.index
)

# Añadir categóricas SIN escalar
X_train_log = pd.concat(

    [
        X_train_scaled_num,
        X_train[X_cat.columns]
    ],

    axis=1
)

X_test_log = pd.concat(

    [
        X_test_scaled_num,
        X_test[X_cat.columns]
    ],

    axis=1
)

In [ ]:
# Modelos

modelos = {

    "Regresión logística": LogisticRegression(
        max_iter=1000
    ),

    "Árbol": DecisionTreeClassifier(

        max_depth=8,

        random_state=42
    ),

    "Random Forest": RandomForestClassifier(

        n_estimators=200,

        max_depth=10,

        random_state=42,

        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(

        n_estimators=200,

        max_depth=6,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        random_state=42
    ),
    "LightGBM": LGBMClassifier(

    n_estimators=200,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    verbose=-1
),

    "CatBoost": CatBoostClassifier(

        iterations=200,

        learning_rate=0.05,

        depth=6,

        verbose=0,

        random_state=42
    )
}

# Evaluación

resultados = []

for nombre, modelo in modelos.items():

    # Regresión logística usa escalado
    if nombre == "Regresión logística":

        modelo.fit(
            X_train_log,
            y_train
        )

        y_pred = modelo.predict(
            X_test_log
        )

        y_prob = modelo.predict_proba(
            X_test_log
        )[:,1]

    # Árboles NO usan escalado
    else:

        modelo.fit(
            X_train,
            y_train
        )

        y_pred = modelo.predict(
            X_test
        )

        y_prob = modelo.predict_proba(
            X_test
        )[:,1]
        
    # Métricas

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    sensibilidad = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    # Matriz confusión
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    especificidad = tn / (tn + fp)

    # Guardar resultados
    resultados.append({

        "Modelo": nombre,

        "Accuracy": accuracy,

        "Precision": precision,

        "Sensibilidad": sensibilidad,

        "Especificidad": especificidad,

        "F1": f1,

        "AUC": auc
    })

# Resultados finales

resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(

    by="AUC",

    ascending=False
)

print(resultados_df)

#### Curvas ROC

In [ ]:

from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(8,6))
for nombre, modelo in modelos.items():
    # Probabilidades
    # Regresión logística usa escalado
    if nombre == "Regresión logística":

        y_prob = modelo.predict_proba(
            X_test_log
        )[:,1]

    # Árboles NO escalados
    else:

        y_prob = modelo.predict_proba(
            X_test
        )[:,1]

    # Curva ROC
    fpr, tpr, _ = roc_curve(
        y_test,
        y_prob
    )

    # AUC
    auc = roc_auc_score(
        y_test,
        y_prob
    )

    # Gráfico ROC

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=f"{nombre} (AUC = {auc:.3f})"
    )
# Línea diagonal (modelo aleatorio)

plt.plot(

    [0,1],
    [0,1],

    linestyle="--",

    linewidth=1
)
# Etiquetas y título

plt.xlabel(
    "1 - Especificidad"
)

plt.ylabel(
    "Sensibilidad"
)

plt.title(
    "Curvas ROC de los modelos de clasificación"
)

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

Importancia de las variables

In [ ]:
rf_model = modelos["Random Forest"]
importancias_rf = pd.DataFrame({

    "Variable": X.columns,

    "Importancia": rf_model.feature_importances_
})

importancias_rf = importancias_rf.sort_values(

    by="Importancia",

    ascending=False
)

print(importancias_rf.head(15))

In [ ]:
top_rf = importancias_rf.head(15)

plt.figure(figsize=(12,8))

sns.barplot(

    data=top_rf,

    x="Importancia",

    y="Variable"
)

plt.title(
    "Importancia de variables - Random Forest"
)

plt.xlabel("Importancia")

plt.ylabel("Variable")

plt.show()

In [ ]:
# Comparación de accuracy según el progreso de la batalla

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Figura
plt.figure(figsize=(10,6))


for nombre, modelo in modelos.items():
    # Predicciones
    # Regresión logística usa datos escalados
    if nombre == "Regresión logística":

        y_pred = modelo.predict(
            X_test_log
        )

    # Árboles NO escalados
    else:

        y_pred = modelo.predict(
            X_test
        )

    # Crear DataFrame de resultados
    df_resultados = pd.DataFrame({

        "Id_Batalla": df.loc[
            test_mask,
            "Id_Batalla"
        ].values,

        "Turno": df.loc[
            test_mask,
            "Turno"
        ].values,

        "Real": y_test.values,

        "Pred": y_pred
    })

    # Turno máximo por batalla
    turno_max = (

        df_resultados
        .groupby("Id_Batalla")["Turno"]
        .max()
        .rename("Turno_max")
    )

    # Unir
    df_resultados = df_resultados.merge(

        turno_max,

        on="Id_Batalla"
    )

    # Porcentaje de progreso de la batalla

    df_resultados["pct_batalla"] = (

        df_resultados["Turno"] /
        df_resultados["Turno_max"]

    ) * 100

    # Crear bins de porcentaje (0-10%, 10-20%, ..., 90-100%)
    df_resultados["pct_bin"] = (

        df_resultados["pct_batalla"] // 10

    ) * 10

    # Accuracy por bin de porcentaje

    accuracy_por_pct = (

        df_resultados

        .groupby("pct_bin")

        .apply(

            lambda x: accuracy_score(
                x["Real"],
                x["Pred"]
            )
        )

        .reset_index(name="Accuracy")

        .sort_values("pct_bin")
    )

 # Gráfico de evolución del accuracy según el progreso de la batalla

    plt.plot(

        accuracy_por_pct["pct_bin"],

        accuracy_por_pct["Accuracy"],

        marker="o",

        markersize=4,

        linewidth=1.5,

        label=nombre
    )

# Etiquetas y título
plt.xlabel(
    "Progreso de la batalla (%)"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Evolución del accuracy según el progreso de la batalla"
)

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()

#### Validación cruzada

In [ ]:
# VALIDACIÓN CRUZADA

from sklearn.model_selection import (
    GroupKFold,
    cross_val_score
)

# GroupKFold para evitar fuga de información entre batallas

gkf = GroupKFold(
    n_splits=5
)

# Evaluar cada modelo con validación cruzada

resultados_cv = []


for nombre, modelo in modelos.items():

    print(f"\n{nombre}")

    # Regresión logística usa escalado
    if nombre == "Regresión logística":

        scores = cross_val_score(

            modelo,

            X_train_log,
            y_train,

            cv=gkf,

            groups=df.loc[
                train_mask,
                "Id_Batalla"
            ],

            scoring="roc_auc",

            n_jobs=-1
        )

    else:

        scores = cross_val_score(

            modelo,

            X_train,
            y_train,

            cv=gkf,

            groups=df.loc[
                train_mask,
                "Id_Batalla"
            ],

            scoring="roc_auc",

            n_jobs=-1
        )

    # Guardar resultados

    resultados_cv.append({

        "Modelo": nombre,

        "AUC media": scores.mean(),

        "Desviación típica": scores.std(),

        "AUC mínima": scores.min(),

        "AUC máxima": scores.max()
    })

    # Resultados por fold

    print("AUC por fold:")

    print(scores)

    print(f"\nAUC media: {scores.mean():.4f}")

    print(f"Desviación típica: {scores.std():.4f}")



In [ ]:
# Crear DataFrame de resultados

cv_df = pd.DataFrame(
    resultados_cv
)

cv_df = cv_df.sort_values(

    by="AUC media",

    ascending=False
)

# Mostrar resultados

print("RESULTADOS VALIDACIÓN CRUZADA")

print(cv_df)


In [ ]:
# Gráfico de comparación de modelos según AUC media

plt.figure(figsize=(10,6))

sns.barplot(

    data=cv_df,

    x="AUC media",

    y="Modelo"
)

plt.xlabel("AUC media")

plt.ylabel("Modelo")

plt.title(
    "Comparación de modelos mediante validación cruzada"
)

plt.xlim(0.5, 1)

plt.grid(True)

plt.tight_layout()

plt.show()

### Otra forma de ingieneria de variables

In [ ]:
variables_numericas = [

    'Turno',

    'Num_Pokemon_J1',
    'Num_Pokemon_J2',

    'Vivos_J1',
    'Vivos_J2',

    'Suma_Vida_J1',
    'Suma_Vida_J2',

    'velocidad_activo_j1',
    'ataque_activo_j1',
    'ataque_esp_activo_j1',
    'defensa_esp_activo_j1',

    'velocidad_activo_j2',
    'ataque_activo_j2',
    'ataque_esp_activo_j2',
    'defensa_esp_activo_j2'
]

variables_categoricas = [

    'tipo_1_activo_j1',
    'tipo_1_activo_j2',

    'Accion_J1',
    'Accion_J2',

    'Primera_Accion'
]

In [ ]:
X_cat = pd.get_dummies(

    df[variables_categoricas],

    drop_first=True
)

X_num = df[variables_numericas]

X = pd.concat(
    [X_num, X_cat],
    axis=1
)

y = (
    df["Ganador"] == "Jugador1"
).astype(int)

groups = df["Id_Batalla"]
print(X.columns.tolist())

In [ ]:
batallas_ids = df["Id_Batalla"].unique()

from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(

    batallas_ids,

    test_size=0.3,

    random_state=42
)

train_mask = df["Id_Batalla"].isin(train_ids)

test_mask = df["Id_Batalla"].isin(test_ids)

X_train = X[train_mask]

X_test = X[test_mask]

y_train = y[train_mask]

y_test = y[test_mask]

print(f"Batallas train: {len(train_ids)}")
print(f"Batallas test: {len(test_ids)}")

print(f"Filas train: {X_train.shape[0]}")
print(f"Filas test: {X_test.shape[0]}")

In [ ]:
# Escalado solo para regresión logística

scaler = StandardScaler()

# Escalar solo numéricas
X_train_scaled_num = scaler.fit_transform(
    X_train[variables_numericas]
)

X_test_scaled_num = scaler.transform(
    X_test[variables_numericas]
)

# Convertir a DataFrame
X_train_scaled_num = pd.DataFrame(

    X_train_scaled_num,

    columns=variables_numericas,

    index=X_train.index
)

X_test_scaled_num = pd.DataFrame(

    X_test_scaled_num,

    columns=variables_numericas,

    index=X_test.index
)

# Añadir categóricas SIN escalar
X_train_log = pd.concat(

    [
        X_train_scaled_num,
        X_train[X_cat.columns]
    ],

    axis=1
)

X_test_log = pd.concat(

    [
        X_test_scaled_num,
        X_test[X_cat.columns]
    ],

    axis=1
)

In [ ]:
# Modelos

modelos = {

    "Regresión logística": LogisticRegression(
        max_iter=1000
    ),

    "Árbol": DecisionTreeClassifier(

        max_depth=8,

        random_state=42
    ),

    "Random Forest": RandomForestClassifier(

        n_estimators=200,

        max_depth=10,

        random_state=42,

        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(

        n_estimators=200,

        max_depth=6,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        random_state=42
    ),
    "LightGBM": LGBMClassifier(

    n_estimators=200,

    learning_rate=0.05,

    max_depth=6,

    subsample=0.8,

    colsample_bytree=0.8,

    random_state=42,

    verbose=-1
),
    "CatBoost": CatBoostClassifier(

        iterations=200,

        learning_rate=0.05,

        depth=6,

        verbose=0,

        random_state=42
    )
    
}

# Evaluación

resultados = []

for nombre, modelo in modelos.items():

    # Regresión logística usa escalado

    if nombre == "Regresión logística":

        modelo.fit(
            X_train_log,
            y_train
        )

        y_pred = modelo.predict(
            X_test_log
        )

        y_prob = modelo.predict_proba(
            X_test_log
        )[:,1]

    else:

        modelo.fit(
            X_train,
            y_train
        )

        y_pred = modelo.predict(
            X_test
        )

        y_prob = modelo.predict_proba(
            X_test
        )[:,1]

    # Métricas
    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred
    )

    sensibilidad = recall_score(
        y_test,
        y_pred
    )

    f1 = f1_score(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    # Matriz confusión
    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    especificidad = tn / (tn + fp)

    # Guardar resultados
    resultados.append({

        "Modelo": nombre,

        "Accuracy": accuracy,

        "Precision": precision,

        "Sensibilidad": sensibilidad,

        "Especificidad": especificidad,

        "F1": f1,

        "AUC": auc
    })
# Resultados finales
resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(

    by="AUC",

    ascending=False
)

print(resultados_df)

#### Curvas ROC

In [ ]:
# Gráfico de curvas ROC

from sklearn.metrics import roc_curve, roc_auc_score

plt.figure(figsize=(8,6))


for nombre, modelo in modelos.items():

    # Probabilidades

    # Regresión logística usa escalado
    if nombre == "Regresión logística":

        y_prob = modelo.predict_proba(
            X_test_log
        )[:,1]

    # Árboles NO escalados
    else:

        y_prob = modelo.predict_proba(
            X_test
        )[:,1]

    # Curva ROC

    fpr, tpr, _ = roc_curve(
        y_test,
        y_prob
    )

    # AUC

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    # Gráfico ROC

    plt.plot(

        fpr,

        tpr,

        linewidth=2,

        label=f"{nombre} (AUC = {auc:.3f})"
    )

# Línea diagonal (modelo aleatorio)

plt.plot(

    [0,1],
    [0,1],

    linestyle="--",

    linewidth=1
)

# Etiquetas y título

plt.xlabel(
    "1 - Especificidad"
)

plt.ylabel(
    "Sensibilidad"
)

plt.title(
    "Curvas ROC de los modelos de clasificación"
)

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

#### Importancia de las variables

In [ ]:
rf_model = modelos["Random Forest"]
importancias_rf = pd.DataFrame({

    "Variable": X.columns,

    "Importancia": rf_model.feature_importances_
})

importancias_rf = importancias_rf.sort_values(

    by="Importancia",

    ascending=False
)

print(importancias_rf.head(15))

In [ ]:
top_rf = importancias_rf.head(15)

plt.figure(figsize=(12,8))

sns.barplot(

    data=top_rf,

    x="Importancia",

    y="Variable"
)

plt.title(
    "Importancia de variables - Random Forest"
)

plt.xlabel("Importancia")

plt.ylabel("Variable")

plt.show()

In [ ]:
# Comparación de accuracy según el progreso de la batalla
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

# Figura
plt.figure(figsize=(10,6))

for nombre, modelo in modelos.items():

    # Predicciones

    # Regresión logística usa datos escalados
    if nombre == "Regresión logística":

        y_pred = modelo.predict(
            X_test_log
        )

    # Árboles NO escalados
    else:

        y_pred = modelo.predict(
            X_test
        )

    # DataFrame de resultados

    df_resultados = pd.DataFrame({

        "Id_Batalla": df.loc[
            test_mask,
            "Id_Batalla"
        ].values,

        "Turno": df.loc[
            test_mask,
            "Turno"
        ].values,

        "Real": y_test.values,

        "Pred": y_pred
    })

    # Turno máximo por batalla

    turno_max = (

        df_resultados
        .groupby("Id_Batalla")["Turno"]
        .max()
        .rename("Turno_max")
    )

    # Unir
    df_resultados = df_resultados.merge(

        turno_max,

        on="Id_Batalla"
    )

    # Porcentaje de progreso de la batalla

    df_resultados["pct_batalla"] = (

        df_resultados["Turno"] /
        df_resultados["Turno_max"]

    ) * 100

    # Crear bins de porcentaje (0-10%, 10-20%, ..., 90-100%)

    df_resultados["pct_bin"] = (

        df_resultados["pct_batalla"] // 10

    ) * 10

    # Accuracy por bin de porcentaje

    accuracy_por_pct = (

        df_resultados

        .groupby("pct_bin")

        .apply(

            lambda x: accuracy_score(
                x["Real"],
                x["Pred"]
            )
        )

        .reset_index(name="Accuracy")

        .sort_values("pct_bin")
    )

    # Gráfico de evolución del accuracy según el progreso de la batalla

    plt.plot(

        accuracy_por_pct["pct_bin"],

        accuracy_por_pct["Accuracy"],

        marker="o",

        markersize=4,

        linewidth=1.5,

        label=nombre
    )

# Etiquetas y título

plt.xlabel(
    "Progreso de la batalla (%)"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Evolución del accuracy según el progreso de la batalla"
)

plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()

#### Validación cruzada

In [ ]:
# Validación cruzada

from sklearn.model_selection import (
    GroupKFold,
    cross_val_score
)

# Definir validación cruzada con GroupKFold para evitar fuga de información entre batallas

gkf = GroupKFold(
    n_splits=5
)

# Evaluar cada modelo con validación cruzada

resultados_cv = []


for nombre, modelo in modelos.items():
    # Regresión logística usa escalado
    print(f"\n{nombre}")

    if nombre == "Regresión logística":

        scores = cross_val_score(

            modelo,

            X_train_log,
            y_train,

            cv=gkf,

            groups=df.loc[
                train_mask,
                "Id_Batalla"
            ],

            scoring="roc_auc",

            n_jobs=-1
        )

    else:

        scores = cross_val_score(

            modelo,

            X_train,
            y_train,

            cv=gkf,

            groups=df.loc[
                train_mask,
                "Id_Batalla"
            ],

            scoring="roc_auc",

            n_jobs=-1
        )

    # Guardar resultados

    resultados_cv.append({

        "Modelo": nombre,

        "AUC media": scores.mean(),

        "Desviación típica": scores.std(),

        "AUC mínima": scores.min(),

        "AUC máxima": scores.max()
    })

    # Mostrar resultados por fold

    print("AUC por fold:")

    print(scores)

    print(f"\nAUC media: {scores.mean():.4f}")

    print(f"Desviación típica: {scores.std():.4f}")



In [ ]:
# Tabla de resultados

cv_df = pd.DataFrame(
    resultados_cv
)

cv_df = cv_df.sort_values(

    by="AUC media",

    ascending=False
)

print("RESULTADOS VALIDACIÓN CRUZADA")

print(cv_df)


In [ ]:
# Gráfico de comparación de modelos según AUC media

plt.figure(figsize=(10,6))

sns.barplot(

    data=cv_df,

    x="AUC media",

    y="Modelo"
)

plt.xlabel("AUC media")

plt.ylabel("Modelo")

plt.title(
    "Comparación de modelos mediante validación cruzada"
)

plt.xlim(0.5, 1)

plt.grid(True)

plt.tight_layout()

plt.show()